# 🎯 Selección de Features

Este notebook implementa múltiples técnicas de selección de características para identificar las variables más relevantes para el modelo.

**Objetivo:** Reducir dimensionalidad y mejorar interpretabilidad del modelo manteniendo el poder predictivo.

## 1. Configuración e Importación de Librerías

In [0]:
# Celda 1: Recepción de Parámetros
schema_name = dbutils.widgets.text("schema_name", "")
schema_name = dbutils.widgets.get("schema_name")

silver_table = dbutils.widgets.text("silver_table", "")
silver_table = dbutils.widgets.get("silver_table")

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, UnivariateFeatureSelector, OneHotEncoder, StandardScaler, PCA
from pyspark.sql import functions as F
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

import pandas as pd

## 2. Cargar el conjunto de datos

Se cargará la tabla `silver_table` para el análisis.

In [0]:
# generar nombre completo de la tabla
def qname(table):
    return f"{schema_name}.{table}"

SILVER_FULL = qname(silver_table)
print("Tabla Silver:", SILVER_FULL)

In [0]:
# Leer datos de la tabla Bronze
lpn_silver = spark.table(SILVER_FULL)

display(lpn_silver.limit(20))

In [0]:
# Dimensiones del dataset
print(f"Filas: {lpn_silver.count()}, Columnas: {len(lpn_silver.columns)}")

### 2.1 Division de datos en train y test

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

target_col = "LABEL_ZONE"
train_frac = 0.8

# Añadir columna aleatoria
df = lpn_silver.withColumn("rand", F.rand())

# Calcular percentil por clase
window = Window.partitionBy(target_col).orderBy("rand")
df = df.withColumn("row_number", F.row_number().over(window))
df = df.withColumn("count_per_class", F.count("*").over(Window.partitionBy(target_col)))
df = df.withColumn("frac", F.col("row_number") / F.col("count_per_class"))

# Split estratificado
train_df = df.filter(F.col("frac") <= train_frac).drop("rand", "row_number", "count_per_class", "frac")
test_df = df.filter(F.col("frac") > train_frac).drop("rand", "row_number", "count_per_class", "frac")

print(f"Train: {train_df.count()} filas")
print(f"Test:  {test_df.count()} filas")

### 2.2 Variable Objetivo

In [0]:
label_indexer = StringIndexer(inputCol="LABEL_ZONE", outputCol="LABEL_ZONE_idx")

## 3. Análisis de Varianza (Variance Threshold)
Eliminar features con varianza muy baja o nula (casi constantes).

In [0]:
# Selecciona solo columnas numéricas
numeric_types = {"int", "bigint", "double", "float", "decimal", "long", "short"}
dtypes = dict(lpn_silver.dtypes)
num_cols = [c for c, t in dtypes.items() if any(t.startswith(nt) for nt in numeric_types)]

# Calcular varianza de cada columna numérica
var_df = lpn_silver.select([
    F.variance(F.col(c)).alias(c) for c in num_cols
])

# Convertir a pandas para visualizar fácilmente
var_pd = var_df.toPandas().T
var_pd.columns = ["variance"]
var_pd = var_pd.sort_values("variance")

# Mostrar top features con menor varianza
print(var_pd.head(10))

# Filtrar columnas con varianza mayor a un umbral (ejemplo: 0.01)
threshold = 0.01
selected_features = var_pd[var_pd["variance"] > threshold].index.tolist()
print(f"Features seleccionadas (varianza > {threshold}): {selected_features}")

## 4. Análisis de Correlación

Identificar y eliminar features altamente correlacionadas (multicolinealidad).

In [0]:
# Selecciona solo columnas numéricas
numeric_types = {"int", "bigint", "double", "float", "decimal", "long", "short"}
dtypes = dict(lpn_silver.dtypes)
num_cols = [c for c, t in dtypes.items() if any(t.startswith(nt) for nt in numeric_types)]

# Ensambla las columnas numéricas en un vector
assembler = VectorAssembler(inputCols=num_cols, outputCol="features_vec")
df_vec = assembler.transform(lpn_silver).select("features_vec")

# Calcula la matriz de correlación de Pearson
corr_matrix = Correlation.corr(df_vec, "features_vec", "pearson").head()[0].toArray()

# Convierte a DataFrame de pandas para visualización
corr_pd = pd.DataFrame(corr_matrix, columns=num_cols, index=num_cols)
print(corr_pd)


In [0]:
import numpy as np

# Definir umbral de correlación alta
threshold = 0.8

# Obtener pares de variables con correlación alta
high_corr_pairs = []

for i in range(len(corr_pd.columns)):
    for j in range(i+1, len(corr_pd.columns)):
        corr_value = corr_pd.iloc[i, j]
        if abs(corr_value) >= threshold:
            high_corr_pairs.append({
                'Variable_1': corr_pd.columns[i],
                'Variable_2': corr_pd.columns[j],
                'Correlacion': corr_value
            })

# Convertir a DataFrame y ordenar por correlación absoluta
high_corr_df = pd.DataFrame(high_corr_pairs)
if not high_corr_df.empty:
    high_corr_df['Correlacion_Abs'] = high_corr_df['Correlacion'].abs()
    high_corr_df = high_corr_df.sort_values('Correlacion_Abs', ascending=False)
    high_corr_df = high_corr_df.drop('Correlacion_Abs', axis=1)
    
    print(f"\n=== Variables con correlación >= {threshold} ===")
    print(f"Total de pares encontrados: {len(high_corr_df)}\n")
    print(high_corr_df.to_string(index=False))
else:
    print(f"\nNo se encontraron pares de variables con correlación >= {threshold}")

## 5. Univariate Feature Selection (Statistical Tests)

Selección basada en tests estadísticos (ANOVA F-value, Mutual Information).

In [0]:
# 1. Indexar la variable objetivo si es string
df2 = label_indexer.fit(train_df).transform(train_df)

# 2. Vectorizar las features numéricas
numeric_types = {"int", "bigint", "double", "float", "decimal", "long", "short"}
dtypes = dict(df2.dtypes)
num_cols = [c for c, t in dtypes.items() if any(t.startswith(nt) for nt in numeric_types)]
assembler = VectorAssembler(inputCols=num_cols, outputCol="features_vec")
df_vec = assembler.transform(df2)

# 3. ANOVA F-test usando UnivariateFeatureSelector
selector = UnivariateFeatureSelector(featuresCol="features_vec", outputCol="selectedFeatures", labelCol="LABEL_ZONE_idx")
selector.setFeatureType("continuous").setLabelType("categorical").setSelectionMode("numTopFeatures").setSelectionThreshold(10)

# Ajustar el selector
model = selector.fit(df_vec)

# Mostrar las features seleccionadas
selected_indices = model.selectedFeatures
print(f"Top {len(selected_indices)} features seleccionadas:")
for idx in selected_indices:
    print(f"  {num_cols[idx]}")

## 6. Feature Importance (Random Forest)

Método embedded: importancia de features usando Random Forest.

### 6.1 Normalización/Escalado

In [0]:
dtypes = dict(train_df.dtypes)
numeric_types = {"int", "bigint", "double", "float", "decimal", "long", "short"}
num_cols = [c for c,t in dtypes.items() if any(t.startswith(nt) for nt in numeric_types)]
print(num_cols)

In [0]:
num_vec = VectorAssembler(inputCols=num_cols, outputCol="num_vec", handleInvalid="keep")
scaler  = StandardScaler(inputCol="num_vec", outputCol="num_std", withMean=True, withStd=True)

### 6.2 One-Hot Encoding 
Se aplicara esta tecnica para features categoricas

In [0]:
# obtener variables categoricas menos LABEL_ZONE
dtypes = dict(train_df.dtypes)
cat_cols = [c for c in train_df.columns if dtypes[c] == "string" and c != "LABEL_ZONE"]
print(cat_cols)

In [0]:
# Index + OneHot por categórica; luego ensamblar con numéricas
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_oh", handleInvalid="keep", dropLast=False) for c in cat_cols]

### 6.3 Ensamblado final de features (numéricas escaladas + categóricas OHE)

In [0]:
final_inputs = ["num_std"] + [f"{c}_oh" for c in cat_cols]
features = VectorAssembler(inputCols=final_inputs, outputCol="features", handleInvalid="keep")

### 6.4 Pipeline

In [0]:
selector_pipe = Pipeline(stages=[label_indexer] + indexers + encoders + [num_vec, scaler, features])
selector_model = selector_pipe.fit(train_df)
df_final = selector_model.transform(train_df)

### 6.5 Modelo

In [0]:
rf = RandomForestClassifier(featuresCol="features", labelCol="LABEL_ZONE_idx", seed=42, numTrees=100)
tmp_model = rf.fit(df_final)

In [0]:
rf_model = tmp_model
importances = rf_model.featureImportances

# Definir k (número de top features a mostrar)
k = 15

# Obtener metadata del vector de features para extraer nombres reales
sample_transformed = tmp_model.transform(df_final.limit(1))
features_metadata = sample_transformed.schema["features"].metadata

# Extraer nombres de todas las features
all_feature_names = {}
if "ml_attr" in features_metadata:
    ml_attr = features_metadata["ml_attr"]
    if "attrs" in ml_attr:
        attrs = ml_attr["attrs"]
        
        # Procesar features numéricas
        if "numeric" in attrs:
            for attr in attrs["numeric"]:
                idx = attr.get("idx")
                name = attr.get("name", f"feature_{idx}")
                # Mapear num_std_X a nombre de columna original
                if name.startswith("num_std_"):
                    num_idx = int(name.split("_")[-1])
                    if num_idx < len(num_cols):
                        name = num_cols[num_idx]
                all_feature_names[idx] = name
        
        # Procesar features binarias (one-hot encoded)
        if "binary" in attrs:
            for attr in attrs["binary"]:
                idx = attr.get("idx")
                name = attr.get("name", f"feature_{idx}")
                all_feature_names[idx] = name

# Top-k índices por importancia
indexed_importances = list(enumerate(importances.toArray()))
indexed_importances.sort(key=lambda x: x[1], reverse=True)

print(f"Top {k} features por importancia:\n")
for rank, (idx, importance) in enumerate(indexed_importances[:k], 1):
    feature_name = all_feature_names.get(idx, f"feature_{idx}")
    print(f"  {rank}. {feature_name}: {importance:.6f}")

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Hacer predicciones en el conjunto de entrenamiento
train_predictions = tmp_model.transform(df_final)

# Evaluar accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="LABEL_ZONE_idx", predictionCol="prediction", metricName="accuracy")
train_accuracy = evaluator.evaluate(train_predictions)
f1 = MulticlassClassificationEvaluator(labelCol="LABEL_ZONE_idx", predictionCol="prediction", metricName="f1")
train_f1 = f1.evaluate(train_predictions)
print(f"Modelo robusto solo con variables numéricas/transformadas -> Accuracy: {train_accuracy:.4f} | F1 Score: {train_f1:.4f}")

## 8. Registrar Pipeline para usar en fase de entrenamiento en base al proceso

In [0]:
# guardar pipeline
pipeline_path = "/tmp/pipeline/transformer"
selector_model.write().overwrite().save(pipeline_path)